# 05b · Folder-grouped development evaluation of the classical baselines

## Purpose
The closed-set validation split saturates (notebook 05): several models classify it perfectly, so it cannot rank models
or show what generalises. This notebook evaluates the same classical models on a **development pool** with folds grouped
by **folder proxy** (class × severity folder, a proxy for an acquisition session), so every recording is predicted by a
model that never saw its folder (normal excepted, see below).

**Data rule.** The pool excludes every reserved test: `split_closed` test, `split_severity` test and `split_speed_block`
test. Known fault classes are used here, which is allowed for this closed-set development exercise; it also means the
classes later withheld in LOCO have influenced method development (see Limitations).

**Note (26 Sep 2026):** the models here used the band-stop-only table (r1). The primary pipeline is now r1cap10.
These 2B.1 results are the **classical benchmark**; no feature redesign around the difficult folders. Model and variant
choices use this folder-grouped development scheme (pool, K = 3); closed-set ties are broken by simplicity and latency;
reserved tests are never used for selection.

## Inputs
`results/baselines/dev_pool_inventory.csv`, `results/dev_folds.csv`, `results/baselines/dev_eval.json` and
`results/scores/dev_r1_<model>.parquet`, written by `scripts/05_baselines.py` (tasks `dev-inventory`, `dev-folds`, `dev`).
The model configuration and recording-level aggregation are identical to Stage 2B.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the settings, <code>metadata.csv</code>, the development results and the fold assignment, and stops if the results were produced under different split assignments than the current ones.</div>

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ml.cache import check_split_hash
from ml.config import load_config
from ml.labels import CLASS6, CLASS10
from ml.paths import DEFAULT_CONFIG, SYSTEM_CODE_DIR

cfg = load_config(DEFAULT_CONFIG)
meta = pd.read_csv(SYSTEM_CODE_DIR / "metadata.csv")
RES = SYSTEM_CODE_DIR / "results" / "baselines"
FIG = SYSTEM_CODE_DIR / "results" / "figures"
SCORES = SYSTEM_CODE_DIR / "results" / "scores"
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
dev = json.loads((RES / "dev_eval.json").read_text())
check_split_hash(dev["provenance"]["split_hash"], meta, "dev_eval.json")
folds = pd.read_csv(SYSTEM_CODE_DIR / "results" / cfg.development.folds_file)
K = dev["provenance"]["k"]
print(f"{dev['provenance']['feature_table']}; pool {dev['provenance']['n_recordings']} recordings, K = {K}")
print(dev["provenance"]["bootstrap"])

## 1. The development pool and the folds
Pool rule (config `development`): `split_closed` ∈ {train, val} AND `split_severity` ≠ test AND `split_speed_block` ≠ test.
K = the minimum number of eligible folder proxies over **fault** classes. Fault folders go to folds within each class
(largest first, to the fold with the fewest recordings of that class); a folder is never split. **Normal is the
exception**: it is a single folder, so its recordings are dealt to the folds individually, stratified by speed, and normal
is evaluated **within-folder**.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Shows the pool inventory (recordings per folder proxy, with their speed range), then the fold assignment: recordings per class in each fold, and which folder proxies each fold holds.</div>

In [ ]:
inv = pd.read_csv(RES / "dev_pool_inventory.csv")
print(f"{inv.recordings.sum()} recordings, speeds {inv.speed_min_hz.min():.1f}–{inv.speed_max_hz.max():.1f} Hz; "
      f"folder proxies per class: {inv.groupby('class10', sort=False).size().to_dict()}")
display(inv.round(1))
display(pd.crosstab(folds.class10, folds.dev_fold).reindex(CLASS10))
folds[folds.class10 != "normal"].groupby(["dev_fold"]).folder.apply(lambda s: ", ".join(sorted(set(s)))).to_frame()

**Limitations.**
- Speeds below 40 Hz only (the speed-block test is reserved).
- Few folder proxies per class: 3 for seven of the nine fault classes, one for normal.
- This exercise uses examples of every class that is later withheld in LOCO. LOCO runs can therefore demonstrate
  fold-specific exclusion during fitting and calibration, but cannot claim those classes never influenced overall method
  development. Fold-safe thresholds alone do not resolve this.
- Folders are **proxies** for acquisition sessions.
- Normal is evaluated **within its single folder**.
- Cleaning settings (burst threshold, band-stop, low-pass, envelope band) were inherited from earlier development and
  are **not re-evaluated per fold**.

## 2. Results per model (recording level)
Out-of-fold: each model is fitted K times, on the other folds, with scalers and SVM calibration refitted inside each
fold; every pool recording is scored exactly once. Intervals: stratified, mixed-unit bootstrap conditional on the fitted
folds (fault folder proxies resampled within class with all their recordings; normal recordings individually). The
number of folder proxies per class is shown with the table; **between-session uncertainty for normal cannot be estimated
(one folder)**.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Builds the per-model table: recording-level macro-F1 and balanced accuracy with their bootstrap intervals, for class10 and class6, window-level macro-F1 for contrast, and the total fitting time over the K folds.</div>

In [ ]:
order = ["lgbm", "rf", "logreg", "svm", "knn", "ensemble"]
fmt = lambda v, ci: f"{v:.3f} [{ci[0]:.3f}, {ci[1]:.3f}]"
rows = []
for m in order:
    r = dev["models"][m]
    rows.append({"model": m,
                 "class10 macro-F1": fmt(r["class10"]["macro_f1"], r["class10"]["macro_f1_ci"]),
                 "class10 balanced acc": fmt(r["class10"]["balanced_accuracy"], r["class10"]["balanced_accuracy_ci"]),
                 "class6 macro-F1": fmt(r["class6"]["macro_f1"], r["class6"]["macro_f1_ci"]),
                 "class6 balanced acc": fmt(r["class6"]["balanced_accuracy"], r["class6"]["balanced_accuracy_ci"]),
                 "window macro-F1 (pooled over all windows)": round(r["window_macro_f1"], 3),
                 "fit s (K folds)": round(r.get("seconds_total", np.nan), 1)})
table = pd.DataFrame(rows).set_index("model")
table.to_csv(RES / "dev_eval_table.csv")
print("folder proxies per class:", dev["provenance"]["folder_proxies_per_class"])
table

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Shows each model's F1 for every class (class10), on all pool recordings scored out-of-fold.</div>

In [ ]:
pd.DataFrame({m: dev["models"][m]["class10"]["per_class_f1"] for m in order}).loc[list(CLASS10)].round(3)

### The hardest folder proxies
Share of each folder's recordings classified correctly (class10), per model, sorted by the mean over models.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Tabulates, for every folder proxy, the share of its recordings each model classifies correctly, and lists the folders with the lowest average.</div>

In [ ]:
acc = pd.DataFrame({m: {f: v["accuracy"] for f, v in dev["models"][m]["class10"]["per_folder_accuracy"].items()}
                    for m in order})
acc["n"] = pd.Series({f: v["n"] for f, v in dev["models"]["lgbm"]["class10"]["per_folder_accuracy"].items()})
acc["mean over models"] = acc[order].mean(axis=1)
hard = acc.sort_values("mean over models")
hard.head(8).round(3)

Where do the recordings of the hardest folders go? Recording-level predictions of LightGBM and SVM:

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Rebuilds recording-level predictions from the saved out-of-fold window scores (same aggregation as everywhere) and counts, for the four hardest folders, which class their recordings were predicted as.</div>

In [ ]:
from ml.evaluation import aggregate_recordings

pool_f = folds.set_index("rec_id").folder


def rec_preds(model: str) -> pd.DataFrame:
    s = pd.read_parquet(SCORES / f"dev_r1_{model}.parquet").sort_values(["rec_id", "window_idx"])
    agg = aggregate_recordings(s[[f"score_{c}" for c in CLASS10]].to_numpy(), s.rec_id.to_numpy(),
                               cfg.baselines.prob_floor)
    return pd.DataFrame({"pred": agg.idxmax(axis=1), "folder": pool_f.loc[agg.index]})


worst = list(hard.index[:4])
pd.concat({m: rec_preds(m).query("folder in @worst").groupby("folder").pred.value_counts().unstack(fill_value=0)
           for m in ["lgbm", "svm"]}, names=["model"]).fillna(0).astype(int)

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Draws class10 confusion matrices for LightGBM and SVM over all pool recordings scored out-of-fold.</div>

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
for ax, m in zip(axes, ["lgbm", "svm"]):
    cm = np.array(dev["models"][m]["class10"]["confusion"])
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(10), CLASS10, rotation=60, ha="right", fontsize=7)
    ax.set_yticks(range(10), CLASS10, fontsize=7)
    for (a, b), v in np.ndenumerate(cm):
        if v:
            ax.text(b, a, v, ha="center", va="center", fontsize=7, color="white" if v > cm.max() / 2 else "black")
    ax.set(title=f"{m}, class10, folder-grouped out-of-fold", xlabel="predicted", ylabel="true")
    ax.grid(False)
fig.tight_layout()
fig.savefig(FIG / "05b_confusion_dev.png")

## 3. Comparisons under the folder-grouped folds (LightGBM)
Each variant is refitted out-of-fold on the same K folds; differences are paired over identical mixed-unit bootstrap
draws. **Interpretation rule:** a drop shows that the model depends on the removed information, not that the information
is an artefact. Results are reported; no decision is made here.
- (a) primary vs the 10 kHz-capped table;
- (b) with vs without the 34 envelope features;
- (c) vibration only (accelerometers and envelope) vs microphone only vs all;
- (d) family permutation importance: within each held-out fold (which contains every class), whole recording blocks are
  shuffled across the held-out recordings, all columns of a family together; the drop in pooled out-of-fold
  recording-level macro-F1, over 5 repeats.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the comparison results and tabulates each variant's recording-level macro-F1 with its interval, next to the paired difference against the full feature set and the share of bootstrap draws in which it is positive.</div>

In [ ]:
cmp_ = json.loads((RES / "dev_compare.json").read_text())
check_split_hash(cmp_["provenance"]["split_hash"], meta, "dev_compare.json")
ci = lambda r: f"{r['class10']['macro_f1']:.3f} [{r['class10']['macro_f1_ci'][0]:.3f}, {r['class10']['macro_f1_ci'][1]:.3f}]"
dfmt = lambda d: f"{d['diff']:+.3f} [{d['ci'][0]:+.3f}, {d['ci'][1]:+.3f}]"
rows = [
    {"variant": "band-stop-only table r1 (144 features)", "class10 macro-F1": ci(cmp_["all_features"]), "paired difference": "",
     "P(diff > 0)": ""},
    {"variant": "(a) capped at 10 kHz, r1cap10 (142 features; primary since 26 Sep)", "class10 macro-F1": ci(cmp_["a_capped"]["capped"]),
     "paired difference": "band-stop-only − capped " + dfmt(cmp_["a_capped"]["primary_minus_capped"]),
     "P(diff > 0)": cmp_["a_capped"]["primary_minus_capped"]["p_positive"]},
    {"variant": "(b) without envelope (110 features)", "class10 macro-F1": ci(cmp_["b_without_envelope"]["without"]),
     "paired difference": "with − without " + dfmt(cmp_["b_without_envelope"]["with_minus_without"]),
     "P(diff > 0)": cmp_["b_without_envelope"]["with_minus_without"]["p_positive"]},
    {"variant": f"(c) vibration only ({cmp_['c_modalities']['n_vibration']} features)",
     "class10 macro-F1": ci(cmp_["c_modalities"]["vibration_only"]),
     "paired difference": "all − vibration " + dfmt(cmp_["c_modalities"]["all_minus_vibration"]),
     "P(diff > 0)": cmp_["c_modalities"]["all_minus_vibration"]["p_positive"]},
    {"variant": f"(c) microphone only ({cmp_['c_modalities']['n_microphone']} features)",
     "class10 macro-F1": ci(cmp_["c_modalities"]["microphone_only"]),
     "paired difference": "all − microphone " + dfmt(cmp_["c_modalities"]["all_minus_microphone"]),
     "P(diff > 0)": cmp_["c_modalities"]["all_minus_microphone"]["p_positive"]},
]
print("folder proxies per class:", cmp_["provenance"]["folder_proxies_per_class"])
pd.DataFrame(rows).set_index("variant")

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Shows per-class F1 for each variant side by side, so it is visible which classes each change affects.</div>

In [ ]:
pd.DataFrame({"all": cmp_["all_features"]["class10"]["per_class_f1"],
              "capped 10 kHz": cmp_["a_capped"]["capped"]["class10"]["per_class_f1"],
              "no envelope": cmp_["b_without_envelope"]["without"]["class10"]["per_class_f1"],
              "vibration only": cmp_["c_modalities"]["vibration_only"]["class10"]["per_class_f1"],
              "microphone only": cmp_["c_modalities"]["microphone_only"]["class10"]["per_class_f1"]}).loc[list(CLASS10)].round(3)

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Plots the family permutation importance: how much pooled out-of-fold macro-F1 drops when one family's columns are shuffled between held-out recordings, with the spread over 5 repeats.</div>

In [ ]:
fam = pd.DataFrame(cmp_["d_family_importance"]).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.barh(fam.family, fam.importance, xerr=fam["std"], color="#dd8452")
ax.set(xlabel="drop in pooled out-of-fold macro-F1", title="Family permutation importance (LightGBM, folder-grouped)")
fig.tight_layout()
fig.savefig(FIG / "05b_family_importance.png")
fam.iloc[::-1].round(3).set_index("family")

## 4. Capped logistic-regression reference (Stage 2C-1)
Logistic regression and LightGBM on the **capped table r1cap10** (the primary pipeline since 26 Sep), the same 2B.1 dev
folds, no retuning, compared pairwise over identical mixed-unit draws. The 0.941 in section 2 is logistic regression on
the **original feature table** (r1, band-stop only).

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the capped comparison and tabulates logistic regression and LightGBM on the capped table with their intervals, next to the paired difference (logistic regression minus LightGBM) for class10 and class6.</div>

In [ ]:
lr = json.loads((RES / "dev_capped_lr.json").read_text())
check_split_hash(lr["provenance"]["split_hash"], meta, "dev_capped_lr.json")
ci2 = lambda r, v: f"{r[v]['macro_f1']:.3f} [{r[v]['macro_f1_ci'][0]:.3f}, {r[v]['macro_f1_ci'][1]:.3f}]"
rows = [{"model (capped table r1cap10)": m, "class10 macro-F1": ci2(lr["models"][m], "class10"),
         "class6 macro-F1": ci2(lr["models"][m], "class6")} for m in ("logreg", "lgbm")]
rows.append({"model (capped table r1cap10)": "logistic regression (original feature table r1, section 2)",
             "class10 macro-F1": ci2(dev["models"]["logreg"], "class10"), "class6 macro-F1": ci2(dev["models"]["logreg"], "class6")})
d10, d6 = lr["logreg_minus_lgbm_class10"], lr["logreg_minus_lgbm_class6"]
print(f"paired, logistic regression − LightGBM (capped): class10 {d10['diff']:+.3f} [{d10['ci'][0]:+.3f}, {d10['ci'][1]:+.3f}], "
      f"P(diff > 0) = {d10['p_positive']:.2f}; class6 {d6['diff']:+.3f} [{d6['ci'][0]:+.3f}, {d6['ci'][1]:+.3f}]")
print("folder proxies per class:", lr["provenance"]["folder_proxies_per_class"])
pd.DataFrame(rows).set_index("model (capped table r1cap10)")

Logistic regression is **a simple reference with competitive development performance**: on the capped table it
scores 0.949 [0.908, 0.983] against LightGBM's 0.903 [0.820, 0.965], and the paired difference is +0.046
[+0.011, +0.095] (class6 +0.040 [+0.009, +0.086]). This is conditional on the fitted folds and a pool with few folder
proxies per class; equivalence or superiority in general is not established.

## Findings (per-model results)
- **Folder-grouped out-of-fold evaluation is not saturated.** Recording-level class10 macro-F1 (804 recordings, K = 3, 95 % mixed-unit interval): SVM 0.943 [0.896, 0.980], ensemble 0.943 [0.894, 0.982], logistic regression 0.941 [0.893, 0.982], k-NN 0.930 [0.881, 0.973], random forest 0.905 [0.831, 0.967], LightGBM 0.898 [0.814, 0.963]. class6: 0.933–0.981. LightGBM window-level macro-F1 is 0.892 **pooled over all windows**; the protocol comparison below reports 0.878 as the **mean of the three fold scores**. Intervals overlap widely; with 3 folder proxies for most fault classes they are wide, and they say nothing about between-session variation for normal (one folder).
- **The errors concentrate in the 20 g bearing folders**, the largest added mass left in the pool: overhang_outer/20g (mean correct over models 0.40), underhang_cage/20g (0.51), overhang_cage/20g (0.53). Their recordings are confused with other bearing classes, cage with outer race and across bearing positions (e.g. LightGBM predicts overhang_outer for 19 of 25 underhang_cage/20g recordings), not with normal or the shaft faults. For LightGBM and random forest, horizontal_misalignment/1.5mm (0.62, 0.57) is also hard.
- The closed-set results were mostly tied; folder-grouped evaluation separates the point estimates, but the individual intervals overlap, and no paired comparison has been made.

## Findings (comparisons, LightGBM; a drop shows dependence, not artefact)
- **(a) Capped at 10 kHz:** 0.903 vs 0.898 for the band-stop-only table; band-stop-only − capped −0.005 [−0.012, +0.001]. On this basis and engineering considerations, r1cap10 became the primary pipeline on 26 Sep 2026. No measurable loss from removing content above 10 kHz on this pool; this does not show that the content is an artefact or that generalisation beyond the pool is equivalent.
- **(b) Envelope features:** without them 0.900; with − without −0.002 [−0.009, +0.003]. These 1–5 kHz envelope features add no measurable overall benefit to this LightGBM setup.
- **(c) Modalities:** vibration only 0.881, microphone only 0.701; all − vibration +0.017 [+0.001, +0.045] (positive in 99 % of draws), all − microphone +0.197 [+0.135, +0.250]. The microphone adds about +0.017 macro-F1 to vibration for LightGBM in this evaluation; whether that evidence is physical or artefactual is unresolved.
- **(d) Family importance:** over_axial 0.224, microphone 0.090, over_radial 0.071, then under/over tangential and under_axial (~0.03); envelope families and under_radial ≈ 0. This is the model's reliance, not a statement about physical vs artefactual evidence.


---
# SEPARATE: evaluation-protocol comparison (for the deck)
**A protocol comparison, not a pure leakage estimate, and not a model result.** The same LightGBM, the same development
pool and K = 3 folds, three ways of choosing what is held out:
- **window-random:** windows assigned to folds at random;
- **recording-grouped:** whole recordings assigned to folds at random;
- **folder-grouped:** whole folder proxies (the development folds above).

**Training sizes are matched exactly:** in every protocol, fold j holds out the same number of windows (window-random)
or recordings (recording-grouped) of each class as folder-grouped fold j, so every protocol trains on the same number
of windows of each class; only *which* windows or recordings are held out differs. Written by
`python scripts/05_baselines.py dev-protocol`.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the protocol comparison and plots window-level macro-F1 for each fold under the three protocols, with the share of held-out windows whose recording also had windows in training.</div>

In [ ]:
prot = json.loads((RES / "dev_protocol.json").read_text())
check_split_hash(prot["provenance"]["split_hash"], meta, "dev_protocol.json")
names = {"window_random": "window-random", "recording_grouped": "recording-grouped", "folder_grouped": "folder-grouped"}
fig, ax = plt.subplots(figsize=(8.5, 4))
for i, (key, label) in enumerate(names.items()):
    s = prot[key]["window_macro_f1_per_fold"]
    ax.scatter(np.full(len(s), i) + np.linspace(-0.08, 0.08, len(s)), s, color=f"C{i}", zorder=3)
    ax.hlines(np.mean(s), i - 0.25, i + 0.25, color=f"C{i}", lw=2)
ax.set_xticks(range(3), [f"{l}\n(held-out windows with their\nrecording in training: "
                         f"{prot[k]['share_of_held_out_windows_whose_recording_is_in_training']:.0%})"
                         for k, l in names.items()], fontsize=8)
ax.set(ylabel="window-level macro-F1 (3 folds)", ylim=(0.75, 1.01),
       title="Evaluation-protocol comparison, LightGBM, matched training sizes")
fig.tight_layout()
fig.savefig(FIG / "05b_protocol_comparison.png")
pd.DataFrame({names[k]: {"window macro-F1 (mean of folds)": prot[k]["window_macro_f1_mean"],
                         "recording macro-F1": prot[k].get("recording_macro_f1", np.nan),
                         "training windows per fold": prot[k]["training_windows_per_fold"]} for k in names}).T

**Protocol comparison.** With identical training sizes, window-level macro-F1 (mean of the three fold scores) is 1.000 when windows are
assigned at random, 0.984 when whole recordings are held out, and 0.878 when whole folder proxies are held out
(0.892 pooled over all windows in section 2) (recording-level 0.986
and 0.898). Each step removes a kind of similarity between training and held-out data: sibling windows of the same
recording, then neighbouring recordings of the same folder. The gap between recording- and folder-grouped is the part
that the closed-set split cannot see.